In [6]:
import io
import os
import glob

folder = '/Users/sebastianx/Corpus Linguistics/BNC/Texts'

try:
    from google.colab import files
    uploaded_files = files.upload()
except ImportError:
    uploaded_files = {}
    for file_path in glob.glob(os.path.join(folder, '*')):
        if os.path.isfile(file_path):
            filename = os.path.basename(file_path)
            uploaded_files[filename] = open(file_path, 'rb').read()

print(f"已加载 {len(uploaded_files)} 个文件：")

已加载 4049 个文件：


In [7]:
import xml.etree.ElementTree as ET
import spacy

# Load the large English model for accurate dependency parsing
nlp = spacy.load('en_core_web_lg')
nlp.max_length = 5000000

# --- Demonstration: xcomp/ccomp = complement of help; acl = modifier on object NP ---
demo_sentences = [
    "the council helped patients wishing to appeal against their detention under the new Act.",
    "Money helps solve problems",
    "Money helps to solve problems",
    "the council helped the patients to appeal against the decision",
]

for sent in demo_sentences:
    doc = nlp(sent)
    help_token = None
    for t in doc:
        if t.lemma_ == 'help' and t.pos_ == 'VERB':
            help_token = t
            break
    print(sent)
    if help_token:
        for child in help_token.children:
            if child.dep_ in {'xcomp', 'ccomp', 'acl'}:
                print(f"  {child.text:10} dep={child.dep_:6} head={child.head.text}")
    print()

the council helped patients wishing to appeal against their detention under the new Act.

Money helps solve problems
  solve      dep=xcomp  head=helps

Money helps to solve problems
  solve      dep=xcomp  head=helps

the council helped the patients to appeal against the decision
  appeal     dep=ccomp  head=helped



In [8]:
# Shared stop-word sets used by get_dep_var and get_comp_lemma.
# STOP_C5: finite verb and modal tags that signal a new clause boundary.
# STOP_POS: interrogative/relative pronouns and subordinating conjunctions.
# STOP_TEXT: lexical items that introduce non-complement structures:

STOP_C5   = {'VVD', 'VVZ', 'VHD', 'VHZ', 'VBD', 'VBZ', 'VM0',
             'VDD', 'VDZ'}
STOP_POS  = {'PNQ', 'CJS', 'CJT'}
STOP_TEXT = {
    # conjunctions / subordinators / relative & interrogative words
    'and', 'or', 'if', 'whether', 'which', 'who', 'whom', 'that', 'instead',
    'whose', 'when', 'where', 'how', 'what',
    # prepositions that introduce PPs (never non-finite complements of help)
    'with', 'by', 'for', 'on', 'about',   # original
    'into', 'after', 'at', 'from',
    'against', 'toward', 'towards',
    'around', 'along', 'across',
    'over', 'under', 'onto', 'upon',
    'during', 'since', 'between', 'behind',
    'throughout', 'within', 'without',
    'beside', 'beneath', 'beyond', 'despite',
}

# BNC tags for "do" as a main verb non-finite forms:
# VDI = infinitive ("to do"), VDB = base form ("help do"), DO0 = auxiliary base form
VB_NONFIN = {'VVI', 'VVB', 'VDI', 'VDB', 'VBI', 'VHB', 'VHI'}

# BNC tags that can form part of an object NP
OBJ_TAGS = {
    'AT0',                        # article (a, an, the)
    'DT0',                        # determiner (this, that, some, any)
    'NN0', 'NN1', 'NN2',          # common nouns
    'NP0',                        # proper noun
    'AJ0', 'AJ1', 'AJC', 'AJS',  # adjectives
    'CRD', 'ORD',                 # cardinal / ordinal numbers
    'PNP', 'PNX', 'PNI',          # pronouns (personal, reflexive, indefinite)
}

# Tags / words that classify the object as a pronoun (PRO)
PRO_TAGS  = {'PNP', 'PNX', 'PNI'}
PRO_WORDS = {'someone', 'anyone', 'everyone', 'nobody', 'somebody',
             'who', 'myself', 'yourself', 'himself', 'herself',
             'themselves', 'ourselves', 'this', 'that'}


def _spacy_dep_of_verb(sentence_text, help_surface, verb_surface):
    """Use spaCy to find the dependency relation of verb_surface relative to help.

    Returns the dep_ label of the first token matching verb_surface that is a
    child of help (xcomp, ccomp) or a grandchild via an object NP (acl).
    Returns None if the relationship cannot be determined.

    Used to filter out acl post-nominal modifiers misidentified as complements.
    e.g. "helped patients wishing to appeal"  → 'wishing' is acl on 'patients'
         "help the committee set up by X ..."  → 'set' is acl on 'committee'
    """
    doc = nlp(sentence_text)
    help_token = None
    for token in doc:
        if token.text.lower() == help_surface.lower() and token.pos_ == 'VERB':
            help_token = token
            break
    if help_token is None:
        return None

    verb_lower = verb_surface.lower()

    # Direct complement of help: xcomp or ccomp
    for child in help_token.children:
        if child.dep_ in ('xcomp', 'ccomp') and child.text.lower() == verb_lower:
            return child.dep_

    # acl on help's direct object NP
    for child in help_token.children:
        if child.dep_ in ('dobj', 'obj', 'oprd'):
            for grandchild in child.children:
                if grandchild.dep_ == 'acl' and grandchild.text.lower() == verb_lower:
                    return 'acl'

    return None


def get_dep_var(elements, help_idx, help_c5, sentence_text=None, help_surface=None):
    """Determine the type of non-finite complement following 'help' using BNC c5 tags.

    Scans up to 30 words to the right of help, stopping at clause boundaries.
    Punctuation elements (tag='c') do not count toward the word limit.
    When sentence_text and help_surface are supplied, spaCy is used to verify
    that candidate BARE and ING verbs are genuine complements (xcomp/ccomp)
    and not acl post-nominal modifiers on help's object NP.
    ING detection uses startswith(('VVG', 'NN1-VVG')); BARE detection uses
    VB_NONFIN or startswith('NN1-VVB') for noun/verb-base ambiguity tags.
    Returns one of: 'TO', 'BARE', 'ING', 'INING', or 'NA'.
    """
    # Guard: allow VV* verb tags, NN1-VVG (gerund/noun), NN1-VVB (base verb/noun)
    if not help_c5 or not help_c5.startswith(('VV', 'NN1-VVG', 'NN1-VVB')):
        return 'NA'
    word_count = 0
    i = help_idx + 1
    while i < len(elements) and word_count < 30:
        elem = elements[i]
        i += 1
        if elem.tag == 's':
            break
        if elem.tag not in ['w', 'c']:
            continue
        c5   = elem.get('c5', '')
        text = (elem.text or '').strip().lower()
        if elem.tag == 'w':
            word_count += 1
        # Stop at sentence-ending punctuation
        if elem.tag == 'c' and elem.get('c5', '') == 'PUN':
            break
        if elem.tag == 'w' and text.endswith((',', '.', '!', '?')):
            break
        # Stop at finite verbs, modals, or subordinating conjunctions
        if elem.tag == 'w' and (c5 in STOP_C5 or c5 in STOP_POS or text in STOP_TEXT):
            break
        # TO infinitive: "to (adv/neg)* VVI/VVB/DO0/VDI/VDB"
        if elem.tag == 'w' and (c5 == 'TO0' or text == 'to'):
            for j in range(i, min(i + 6, len(elements))):
                nxt = elements[j]
                if nxt.tag != 'w':
                    continue
                nxt_c5 = nxt.get('c5', '')
                if nxt_c5 in {'AV0', 'XX0', 'TO0'}:
                    continue
                if nxt_c5 in VB_NONFIN or nxt_c5.startswith('NN1-VVB'):
                    return 'TO'
                break
            break
        # INING: "in (det/pron/noun)* VVG"
        if elem.tag == 'w' and text == 'in':
            for j in range(i, min(i + 4, len(elements))):
                nxt = elements[j]
                if nxt.tag != 'w':
                    continue
                if nxt.get('c5', '').startswith(('VVG', 'NN1-VVG')):
                    return 'INING'
                if nxt.get('c5', '') in {'AT0', 'DT0', 'PNP', 'NN0', 'NN1', 'NN2'}:
                    continue
                break
            break  # 'in' didn't lead to INING → stop scanning (prevents in+NP+VVG false positives)
        # ING: catches VVG-AJ0 (VVG primary) and NN1-VVG (gerund/noun ambiguity)
        if elem.tag == 'w' and c5.startswith(('VVG', 'NN1-VVG')):
            if sentence_text and help_surface:
                dep = _spacy_dep_of_verb(sentence_text, help_surface,
                                         (elem.text or '').strip())
                if dep == 'acl':
                    continue
            return 'ING'
        # BARE: bare infinitive — also catches NN1-VVB (noun/base-verb ambiguity)
        if elem.tag == 'w' and (c5 in VB_NONFIN or c5.startswith('NN1-VVB')):
            if sentence_text and help_surface:
                dep = _spacy_dep_of_verb(sentence_text, help_surface,
                                         (elem.text or '').strip())
                if dep == 'acl':
                    continue
            return 'BARE'
    return 'NA'


def get_comp_lemma(elements, help_idx, help_c5, sentence_text=None, help_surface=None):
    """Return the verb lemma of the non-finite complement clause following 'help'.

    Mirrors the search logic of get_dep_var but returns the BNC headword (hw
    attribute) of the identified verb rather than its construction category.
    Punctuation tokens (tag='c') are skipped and do not count as words.
    acl verbs (ING and BARE) are filtered out via spaCy when sentence_text is supplied.
    ING detection uses startswith(('VVG', 'NN1-VVG')); BARE also catches NN1-VVB.
    Returns 'NA' if no non-finite complement is found within 30 words.
    """
    if not help_c5 or not help_c5.startswith(('VV', 'NN1-VVG', 'NN1-VVB')):
        return 'NA'

    def lemma_of(elem):
        hw = elem.get('hw', '').strip().lower()
        return hw if hw else (elem.text or '').strip().lower()

    word_count = 0
    i = help_idx + 1
    while i < len(elements) and word_count < 30:
        elem = elements[i]
        i += 1
        if elem.tag == 's':
            break
        if elem.tag not in ['w', 'c']:
            continue
        c5   = elem.get('c5', '')
        text = (elem.text or '').strip().lower()
        if elem.tag == 'w':
            word_count += 1
        if elem.tag == 'c' and elem.get('c5', '') == 'PUN':
            break
        if elem.tag == 'w' and text.endswith((',', '.', '!', '?')):
            break
        if elem.tag == 'w' and (c5 in STOP_C5 or c5 in STOP_POS or text in STOP_TEXT):
            break
        # TO infinitive: return lemma of the infinitive verb
        if elem.tag == 'w' and (c5 == 'TO0' or text == 'to'):
            for j in range(i, min(i + 6, len(elements))):
                nxt = elements[j]
                if nxt.tag != 'w':
                    continue
                nxt_c5 = nxt.get('c5', '')
                if nxt_c5 in {'AV0', 'XX0', 'TO0'}:
                    continue
                if nxt_c5 in VB_NONFIN or nxt_c5.startswith('NN1-VVB'):
                    return lemma_of(nxt)
                break
            break
        # INING: return lemma of the -ing verb
        if elem.tag == 'w' and text == 'in':
            for j in range(i, min(i + 4, len(elements))):
                nxt = elements[j]
                if nxt.tag != 'w':
                    continue
                if nxt.get('c5', '').startswith(('VVG', 'NN1-VVG')):
                    return lemma_of(nxt)
                if nxt.get('c5', '') in {'AT0', 'DT0', 'PNP', 'NN0', 'NN1', 'NN2'}:
                    continue
                break
            break  # 'in' didn't lead to INING → stop scanning
        # ING: catches VVG-AJ0 and NN1-VVG; verify with spaCy
        if elem.tag == 'w' and c5.startswith(('VVG', 'NN1-VVG')):
            if sentence_text and help_surface:
                dep = _spacy_dep_of_verb(sentence_text, help_surface,
                                         (elem.text or '').strip())
                if dep == 'acl':
                    continue
            return lemma_of(elem)
        # BARE: also catches NN1-VVB; verify with spaCy
        if elem.tag == 'w' and (c5 in VB_NONFIN or c5.startswith('NN1-VVB')):
            if sentence_text and help_surface:
                dep = _spacy_dep_of_verb(sentence_text, help_surface,
                                         (elem.text or '').strip())
                if dep == 'acl':
                    continue
            return lemma_of(elem)
    return 'NA'


def extract_bnc_sentence(elements, help_idx):
    """Reconstruct the plain text of the BNC sentence containing the help token."""
    sent_start = 0
    for k in range(help_idx, -1, -1):
        if elements[k].tag == 's':
            sent_start = k
            break
    parts = []
    for k in range(sent_start, len(elements)):
        elem = elements[k]
        if k > sent_start and elem.tag == 's':
            break
        if elem.tag in ['w', 'c'] and elem.text:
            parts.append(elem.text)
    return ' '.join(parts)


def get_interv_words(elements, help_idx, help_c5, sentence_text=None, help_surface=None):
    """Return the number of words between help and its non-finite complement verb.

    Uses the same BNC c5 tag scanning as get_dep_var.
    intervening_words = (words between help and 'to') + (adverbs/negation between 'to' and verb)
    i.e. equivalent to  i + j - 2  where i = offset to 'to', j = offset from 'to' to verb (1-indexed).

    For BARE/ING/INING: words strictly between help and the complement verb.
    acl post-nominal modifiers (ING and BARE) are skipped via spaCy.
    ING detection uses startswith(('VVG', 'NN1-VVG')); BARE also catches NN1-VVB.
    Returns an integer, or 'NA' if no complement is found.
    """
    if not help_c5 or not help_c5.startswith(('VV', 'NN1-VVG', 'NN1-VVB')):
        return 'NA'
    word_count = 0
    i = help_idx + 1
    while i < len(elements) and word_count < 30:
        elem = elements[i]
        i += 1
        if elem.tag == 's':
            break
        if elem.tag not in ['w', 'c']:
            continue
        c5   = elem.get('c5', '')
        text = (elem.text or '').strip().lower()
        if elem.tag == 'w':
            word_count += 1
        if elem.tag == 'c' and elem.get('c5', '') == 'PUN':
            break
        if elem.tag == 'w' and text.endswith((',', '.', '!', '?')):
            break
        if elem.tag == 'w' and (c5 in STOP_C5 or c5 in STOP_POS or text in STOP_TEXT):
            break
        # TO: words_before_to + adverbs/negation between 'to' and verb
        if elem.tag == 'w' and (c5 == 'TO0' or text == 'to'):
            words_before_to = word_count - 1
            adv_count = 0
            for j in range(i, min(i + 6, len(elements))):
                nxt = elements[j]
                if nxt.tag != 'w':
                    continue
                nxt_c5 = nxt.get('c5', '')
                if nxt_c5 in {'AV0', 'XX0', 'TO0'}:
                    adv_count += 1
                    continue
                if nxt_c5 in VB_NONFIN or nxt_c5.startswith('NN1-VVB'):
                    return words_before_to + adv_count
                break
            break
        # INING: words before 'in' + NP words between 'in' and gerund
        if elem.tag == 'w' and text == 'in':
            words_before_in = word_count - 1
            np_count = 0
            for j in range(i, min(i + 4, len(elements))):
                nxt = elements[j]
                if nxt.tag != 'w':
                    continue
                if nxt.get('c5', '').startswith(('VVG', 'NN1-VVG')):
                    return words_before_in + np_count
                if nxt.get('c5', '') in {'AT0', 'DT0', 'PNP', 'NN0', 'NN1', 'NN2'}:
                    np_count += 1
                    continue
                break
            break  # 'in' didn't lead to INING → stop scanning
        # ING: catches VVG-AJ0 and NN1-VVG; verify with spaCy, skip if acl
        if elem.tag == 'w' and c5.startswith(('VVG', 'NN1-VVG')):
            if sentence_text and help_surface:
                dep = _spacy_dep_of_verb(sentence_text, help_surface,
                                         (elem.text or '').strip())
                if dep == 'acl':
                    continue
            return word_count - 1
        # BARE: also catches NN1-VVB; verify with spaCy, skip if acl
        if elem.tag == 'w' and (c5 in VB_NONFIN or c5.startswith('NN1-VVB')):
            if sentence_text and help_surface:
                dep = _spacy_dep_of_verb(sentence_text, help_surface,
                                         (elem.text or '').strip())
                if dep == 'acl':
                    continue
            return word_count - 1
    return 'NA'


def get_obj_info(elements, help_idx, help_c5, sentence_text=None, help_surface=None):
    """Return object NP info for the complement clause of help.

    Only call this when DepVar is not 'NA'. Scans using the same logic as
    get_dep_var, collecting NP words between help and the complement marker.

    Returns (obj_presence, obj_type, obj_length, obj_head):
      obj_presence : 'Yes' or 'No'
      obj_type     : 'PRO' | 'NP' | 'NA'
      obj_length   : int (word count of object NP) | 'NA'
      obj_head     : str (lemma of head word) | 'NA'

    PRO: single pronoun/pronoun-equivalent (PNP, PNX, PNI tags; someone, myself, this, etc.)
    NP head: spaCy dobj/obj child of help (BNC hw as fallback).
    """
    if not help_c5 or not help_c5.startswith(('VV', 'NN1-VVG', 'NN1-VVB')):
        return ('NA', 'NA', 'NA', 'NA')

    obj_elems = []

    word_count = 0
    i = help_idx + 1
    while i < len(elements) and word_count < 30:
        elem = elements[i]
        i += 1
        if elem.tag == 's':
            break
        if elem.tag not in ['w', 'c']:
            continue
        c5   = elem.get('c5', '')
        text = (elem.text or '').strip().lower()
        if elem.tag == 'w':
            word_count += 1

        # ── stop conditions ───────────────────────────────────────────────
        if elem.tag == 'c' and elem.get('c5', '') == 'PUN':
            break
        if elem.tag == 'w' and text.endswith((',', '.', '!', '?')):
            break
        if elem.tag == 'w' and (c5 in STOP_C5 or c5 in STOP_POS or text in STOP_TEXT):
            break

        # ── complement markers: stop collecting the object ─────────────────
        if elem.tag == 'w' and (c5 == 'TO0' or text == 'to'):
            break
        if elem.tag == 'w' and text == 'in':
            break

        # ING: catches VVG-AJ0 and NN1-VVG; acl → skip, else → stop
        if elem.tag == 'w' and c5.startswith(('VVG', 'NN1-VVG')):
            if sentence_text and help_surface:
                dep = _spacy_dep_of_verb(sentence_text, help_surface,
                                         (elem.text or '').strip())
                if dep == 'acl':
                    continue
            break

        # BARE: also catches NN1-VVB; acl → skip, else → stop
        if elem.tag == 'w' and (c5 in VB_NONFIN or c5.startswith('NN1-VVB')):
            if sentence_text and help_surface:
                dep = _spacy_dep_of_verb(sentence_text, help_surface,
                                         (elem.text or '').strip())
                if dep == 'acl':
                    continue
            break

        # ── collect NP words ───────────────────────────────────────────────
        if elem.tag == 'w':
            if not obj_elems and c5 in {'AV0', 'XX0'}:
                continue
            if c5 in OBJ_TAGS or text in PRO_WORDS:
                obj_elems.append(elem)

    # ── no object words found ──────────────────────────────────────────────
    if not obj_elems:
        return ('No', 'NA', 'NA', 'NA')

    # ── classify PRO vs NP ────────────────────────────────────────────────
    head_c5   = obj_elems[0].get('c5', '')
    head_form = (obj_elems[0].text or '').strip().lower()
    is_pro = (len(obj_elems) == 1 and
              (head_c5 in PRO_TAGS or head_form in PRO_WORDS))
    obj_type   = 'PRO' if is_pro else 'NP'
    obj_length = len(obj_elems)

    # ── object head ───────────────────────────────────────────────────────
    if is_pro:
        obj_head = (obj_elems[0].text or '').strip().lower()
    else:
        obj_head = 'NA'
        if sentence_text and help_surface:
            doc = nlp(sentence_text)
            help_tok = None
            for token in doc:
                if token.text.lower() == help_surface.lower() and token.pos_ == 'VERB':
                    help_tok = token
                    break
            if help_tok:
                for child in help_tok.children:
                    if child.dep_ in ('dobj', 'obj'):
                        obj_head = child.text.lower()
                        break
        if obj_head == 'NA':
            for e in reversed(obj_elems):
                ec5 = e.get('c5', '')
                if ec5.startswith('NN') or ec5.startswith('NP'):
                    obj_head = (e.text or '').strip().lower()
                    break
            else:
                obj_head = (obj_elems[-1].text or '').strip().lower()

    return ('Yes', obj_type, obj_length, obj_head)



In [9]:
import nltk
from nltk.corpus import wordnet as wn
nltk.download('wordnet')   # uncomment on first run if needed


def get_left_vars(elements, help_idx, help_c5):
    """Return (voice, preceding_to, polarity) by scanning up to 5 words left of help.

    voice        : 'Active' | 'Passive' | 'NA'
    preceding_to : 'NOtoBefore' | 'YEStoBefore' | 'NA'
    polarity     : 'POS' | 'NEG' | 'NA'

    Passive = any be-verb (VB*) immediately preceding help, AND help itself is VVN.
    PrecedingTo = TO0 tag or text "to" within 5 words left.
    Polarity = NEG if a negation word is within 5 words left, else POS.
    Stops at sentence boundary (<s> tag), sentence-ending punctuation, or a
    finite/modal verb (STOP_C5) — prevents scanning into a superordinate clause.
    """
    if not help_c5 or not help_c5.startswith(('VV', 'NN1-VVG', 'NN1-VVB')):
        return ('NA', 'NA', 'NA')

    NEG_WORDS = {'not', "n't", 'nor', 'never', 'hardly', 'scarcely', 'barely',
                 'no', 'nobody', 'nothing', 'nowhere'}

    voice        = 'Active'
    preceding_to = 'NOtoBefore'
    polarity     = 'POS'

    words_found = 0
    i = help_idx - 1
    while i >= 0 and words_found < 5:
        elem = elements[i]
        i -= 1
        # Sentence boundary
        if elem.tag == 's':
            break
        if elem.tag not in ['w', 'c']:
            continue
        text = (elem.text or '').strip().lower()
        # Hard punctuation boundary
        if elem.tag == 'c' and text in {'.', '!', '?'}:
            break
        if elem.tag == 'c':
            continue
        # It's a word
        c5 = elem.get('c5', '')
        words_found += 1
        # Voice: be-verb (VB*) + help is VVN = passive
        if c5.startswith('VB') and help_c5 == 'VVN':
            voice = 'Passive'
        # Preceding to (only within 2 words of help)
        if words_found <= 2 and (c5 == 'TO0' or text == 'to'):
            preceding_to = 'YEStoBefore'
        # Polarity
        if text in NEG_WORDS:
            polarity = 'NEG'
        # Clause boundary: finite or modal verb stops the scan
        if c5 in STOP_C5:
            break

    return (voice, preceding_to, polarity)


# ── Animacy lookup ─────────────────────────────────────────────────────────
ANIMATE_HYPERNYMS = {'person.n.01', 'animal.n.01', 'organism.n.01'}

# Pronouns whose animacy is fixed without WordNet
ANIMATE_PROS  = {'i', 'we', 'you', 'he', 'she', 'they', 'who', 'whom',
                 'someone', 'somebody', 'anyone', 'anybody',
                 'everyone', 'everybody', 'nobody', 'noone',
                 'myself', 'yourself', 'himself', 'herself',
                 'themselves', 'ourselves', 'oneself'}
INANIMATE_PROS = {'it', 'this', 'that', 'which', 'what',
                  'nothing', 'everything', 'anything', 'something'}

def get_animacy_of(word):
    """Return 'animate', 'inanimate', or 'NA' for a subject head word."""
    w = word.lower()
    if w in ANIMATE_PROS:    return 'animate'
    if w in INANIMATE_PROS:  return 'inanimate'
    if w in {'null', 'na'}:  return 'NA'
    synsets = wn.synsets(w, pos=wn.NOUN)
    if not synsets:
        return 'NA'
    for syn in synsets:
        for hyp in syn.closure(lambda s: s.hypernyms()):
            if hyp.name() in ANIMATE_HYPERNYMS:
                return 'animate'
    return 'inanimate'


# ── BNC c5 tags to skip when scanning left for subject ──────────────────────
SUBJ_SKIP_C5 = {
    'AV0', 'XX0',                                        # adverbs, negation
    'VM0',                                               # modals
    'VH0', 'VHD', 'VHZ', 'VHI', 'VHG', 'VHN',          # have
    'VB0', 'VBD', 'VBZ', 'VBI', 'VBG', 'VBN',          # be
    'VDD', 'VDZ', 'VDI', 'VDG', 'VDN',                  # do
    'PUN', 'PUL', 'PUR',                                 # punctuation
}


def _spacy_subj_token(sentence_text, help_surface):
    """Return the spaCy subject token of help, or None.

    Handles VP coordination: 'She came and helped me' → returns token 'She'.
    """
    doc = nlp(sentence_text)
    help_tok = None
    for token in doc:
        if token.text.lower() == help_surface.lower() and token.pos_ == 'VERB':
            help_tok = token
            break
    if help_tok is None:
        return None
    # Direct nsubj / nsubjpass
    for child in help_tok.children:
        if child.dep_ in ('nsubj', 'nsubjpass', 'expl'):
            return child
    # VP coordination: help is conj of head verb → shares head's subject
    if help_tok.dep_ == 'conj':
        for child in help_tok.head.children:
            if child.dep_ in ('nsubj', 'nsubjpass'):
                return child
    return None


def get_subj_info(elements, help_idx, help_c5, sentence_text=None, help_surface=None):
    """Return (subj_type, subj_head, animacy) for the subject of 'help'.

    Scans up to 30 words to the left; handles VP coordination.

    subj_type : 'NP' | 'It' | 'PRO' | 'NULL' | 'NA'
    subj_head : head-word string | 'NULL' | 'NA'
    animacy   : 'animate' | 'inanimate' | 'NA'

    NULL — no overt subject (imperative, diary style, non-finite complement)
    NA   — help is not functioning as a main verb
    """
    # Not a verbal use
    if not help_c5 or not help_c5.startswith(('VV', 'NN1-VVG', 'NN1-VVB')):
        return ('NA', 'NA', 'NA')

    # ── Primary: spaCy dependency parse ──────────────────────────────────
    # Run spaCy first for ALL verbal forms — it handles question inversion
    # (has it helped?), passive, modal, progressive, and control structures.
    spacy_tok = None
    if sentence_text and help_surface:
        spacy_tok = _spacy_subj_token(sentence_text, help_surface)

    # For non-finite forms (VVI/VVG/VVN): if spaCy found no subject,
    # use FINITE_AUX check to decide between bare non-finite (→ NULL)
    # and constructions with an overt subject (→ try BNC scan).
    # Bare non-finite: to-infinitive, gerund, control/raising → NULL.
    # With finite auxiliary: progressive, passive, modal, perfect → BNC scan.
    if help_c5 in {'VVI', 'VVG', 'VVN'} and spacy_tok is None:
        FINITE_AUX = {'VBZ', 'VBD', 'VBP', 'VBR', 'VM0',
                      'VHZ', 'VHD', 'VHP', 'VDZ', 'VDD'}
        AUX_NONFIN = {'VBG', 'VBN', 'VBI', 'VHG', 'VHN', 'VHI'}
        # Also skip pronouns: question inversion puts subject between aux and verb
        # e.g. "has it helped?" → scan: it(PNP) → skip → has(VHZ) → found ✓
        SKIP = {'AV0', 'XX0', 'PNP', 'PNI', 'PNX'} | AUX_NONFIN
        has_finite_aux = False
        for k in range(help_idx - 1, max(help_idx - 7, -1), -1):
            e = elements[k]
            if e.tag == 's':
                break
            if e.tag != 'w':
                continue
            ec5 = e.get('c5', '')
            if ec5 in FINITE_AUX:
                has_finite_aux = True
                break
            if ec5 not in SKIP and not ec5.startswith('AV'):
                break
        if not has_finite_aux:
            return ('NULL', 'NULL', 'NA')
        # has_finite_aux but spaCy failed → fall through to BNC scan

    # Process spaCy result (common path for all verbal forms)
    if spacy_tok is not None:
        text  = spacy_tok.text.lower()
        lemma = spacy_tok.lemma_.lower() if spacy_tok.lemma_ != '-PRON-' else text
        if text == 'it':
            return ('It', 'it', get_animacy_of('it'))
        if spacy_tok.pos_ == 'PRON' or text in {'who', 'which', 'that'}:
            return ('PRO', text, get_animacy_of(text))
        return ('NP', lemma, get_animacy_of(lemma))

    # ── Fallback: BNC c5-tag left scan ───────────────────────────────────
    left_words = []
    i, wc = help_idx - 1, 0
    while i >= 0 and wc < 30:
        elem = elements[i]; i -= 1
        if elem.tag == 's':
            break
        if elem.tag not in ['w', 'c']:
            continue
        t = (elem.text or '').strip().lower()
        if elem.tag == 'c' and t in {'.', '!', '?'}:
            break
        if elem.tag == 'w':
            left_words.insert(0, elem); wc += 1

    if not left_words:
        return ('NULL', 'NULL', 'NA')   # nothing to the left → imperative

    j = len(left_words) - 1
    while j >= 0:
        elem = left_words[j]
        c5   = elem.get('c5', '')
        text = (elem.text or '').strip().lower()

        # Skip auxiliaries, modals, adverbs, negation
        if c5 in SUBJ_SKIP_C5 or c5.startswith(('VH', 'VB', 'VM', 'VD')):
            j -= 1; continue

        # VP coordination: "and/or/but" → skip the preceding verb phrase
        if text in {'and', 'or', 'but'} or c5 == 'CJC':
            j -= 1
            while j >= 0:
                e   = left_words[j]
                ec5 = e.get('c5', '')
                et  = (e.text or '').strip().lower()
                if (ec5.startswith(('VV', 'VH', 'VB', 'VM', 'VD')) or
                        ec5 in SUBJ_SKIP_C5 or et in {'and', 'or', 'but', ','}):
                    j -= 1; continue
                break
            continue

        # Pronoun
        if c5 in {'PNP', 'PNI', 'PNX'} or text in {'who', 'whom', 'which', 'that'}:
            if text == 'it':
                return ('It', 'it', get_animacy_of('it'))
            return ('PRO', text, get_animacy_of(text))

        # Noun (common or proper) or gerund acting as subject
        if c5.startswith(('NN', 'NP')) or c5.startswith('VVG'):
            hw = elem.get('hw', '').strip().lower() or text
            return ('NP', hw, get_animacy_of(hw))

        break   # unexpected tag → stop

    return ('NULL', 'NULL', 'NA')


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/sebastianx/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [10]:
# Create KWIC concordance lines for "help" from our XML files

# Output lists — ordered to match the final spreadsheet column layout:
# Hit | KWIC | Form | POS | DepVar | CompLemma | IntervWords |
# ObjPresence | ObjType | ObjLength | ObjHead |
# Voice | PrecedingTo | Polarity |
# SubjType | SubjHead | Animacy |
# TextID | Year | Genre | Subgenre | Mode | Variety | Corpus |
# SpeakerID | SpeakerGender | SpeakerAge | SpeakerSoc
hits         = []
help_KWIC    = []
help_form    = []
help_pos     = []
voice_list    = []
prec_to_list  = []
polarity_list = []
subj_type_list = []
subj_head_list = []
animacy_list   = []
dep_var      = []
comp_lemma   = []
interv_words = []
obj_presence  = []
obj_type_list = []
obj_length_list = []
obj_head_list = []
file_id      = []
year         = []
genre        = []
subgenre     = []
mode         = []
variety      = []
corpus       = []
speaker_id     = []
speaker_gender = []
speaker_age    = []
soc            = []

hit = 1
max_buffer = 30  # for left context, keep last 30 words

# BNC age group codes → midpoint of age range
age_midpoint = {
    'Ag0': 'unknown',
    'Ag1': 7,    # 0-14
    'Ag2': 20,   # 15-24
    'Ag3': 30,   # 25-34
    'Ag4': 40,   # 35-44
    'Ag5': 55,   # 45+
    'X':   'unknown',
}

XML_NS = '{http://www.w3.org/XML/1998/namespace}'


for file, content in uploaded_files.items():
    tree = ET.parse(io.BytesIO(content))

    # --- (1) File-level metadata ---

    id = file.replace('.xml', '')

    classcode = tree.find('.//classCode')
    meta = classcode.text

    creation = tree.find('.//creation')
    yearvalue = creation.get('date')
    if yearvalue == '0000':
        yearvalue = '1990'

    if meta[0] == "W":
        modevalue     = "Written"
        wstext_element = tree.find('.//wtext') or tree.find('.//stext')
        genrevalue    = wstext_element.get('type') if wstext_element is not None else "NA"
        subgenrevalue = meta[2:]
        spk_id, spk_gender, spk_age, spk_soc = "NA", "NA", "NA", "NA"

    elif meta[0] == "S":
        modevalue     = "Spoken"
        genrevalue    = "NA"
        subgenrevalue = "NA"
        person = tree.find('.//person')
        if person is not None:
            spk_id     = person.get(f'{XML_NS}id', 'NA')
            spk_gender = person.get('sex', 'NA')
            raw_age    = person.get('ageGroup', 'Ag0')
            spk_age    = age_midpoint.get(raw_age, 'unknown')
            spk_soc    = person.get('soc', 'NA')
        else:
            spk_id, spk_gender, spk_age, spk_soc = "NA", "NA", "unknown", "NA"

    else:
        modevalue     = "Unknown"
        wstext_element = tree.find('.//wtext') or tree.find('.//stext')
        genrevalue    = wstext_element.get('type') if wstext_element is not None else "NA"
        subgenrevalue = meta[2:]
        spk_id, spk_gender, spk_age, spk_soc = "NA", "NA", "NA", "NA"

    # --- (2) KWIC concordance & annotation ---

    left_context = []
    collecting_right = False
    right_context = []
    current_right_context_words = 0

    all_elems = list(tree.iter())

    for elem_idx, elem in enumerate(all_elems):

        # Collect right context for the previous hit
        if collecting_right:
            if elem.tag in ['w', 'c']:
                right_context.append(elem.text)
                if elem.tag == 'w':
                    current_right_context_words += 1
                    if current_right_context_words >= 60:
                        collecting_right = False
                        right_KWIC = ' '.join(filter(None, right_context))
                        help_KWIC.append(f"{left_KWIC} @{help_word}@ {right_KWIC}")

        # Build rolling left context buffer (30 words)
        if elem.tag in ['w', 'c'] and elem.text and not elem.text.lower().startswith('help'):
            left_context.append(elem.text)
            if len(left_context) > max_buffer:
                left_context.pop(0)

        # Found a "help" hit
        if elem.tag == 'w' and elem.text and elem.text.lower().startswith('help'):

            # Store right context of any preceding hit still being collected
            if collecting_right:
                right_KWIC = ' '.join(filter(None, right_context))
                help_KWIC.append(f"{left_KWIC} @{help_word}@ {right_KWIC}")

            left_KWIC = ' '.join(left_context)
            help_word = elem.text.strip()
            help_c5   = elem.get('c5')

            # ── Verbal vs non-verbal gate ─────────────────────────────────
            # VV*         → always verbal
            # NN1-VVG/VVB → ambiguous: use spaCy pos_ to decide
            # Everything else (NN1, NN2, …) → non-verbal, all vars → NA
            if help_c5 and help_c5.startswith('VV'):
                is_verbal = True
            elif help_c5 in ('NN1-VVG', 'NN1-VVB'):
                sentence_text = extract_bnc_sentence(all_elems, elem_idx)
                is_verbal = False
                for _tok in nlp(sentence_text):
                    if _tok.text.lower() == help_word.lower():
                        is_verbal = (_tok.pos_ == 'VERB')
                        break
            else:
                is_verbal = False

            # ── Context-based noun override ───────────────────────────────
            # Regardless of tag, treat as noun if diagnostic context words appear
            # within 2 words left: self / need / outside
            # within 2 words right: from
            if is_verbal:
                _left2  = [(e.text or '').strip().lower()
                           for e in all_elems[max(0, elem_idx-4):elem_idx]
                           if e.tag == 'w'][-2:]
                _right2 = [(e.text or '').strip().lower()
                           for e in all_elems[elem_idx+1:elem_idx+5]
                           if e.tag == 'w'][:2]
                if (any(w in {'self', 'need', 'outside'} for w in _left2) or
                        any(w in {'from'} for w in _right2)):
                    is_verbal = False

            if is_verbal:
                sentence_text = extract_bnc_sentence(all_elems, elem_idx)

                lv, pt, po = get_left_vars(all_elems, elem_idx, help_c5)

                st, sh, sa = get_subj_info(all_elems, elem_idx, help_c5,
                                           sentence_text, help_word)

                dv = get_dep_var(all_elems, elem_idx, help_c5,
                                 sentence_text, help_word)

                cl = get_comp_lemma(all_elems, elem_idx, help_c5,
                                    sentence_text, help_word)

                iw = get_interv_words(all_elems, elem_idx, help_c5,
                                      sentence_text, help_word)

                if dv != 'NA':
                    op, ot, ol, oh = get_obj_info(all_elems, elem_idx, help_c5,
                                                   sentence_text, help_word)
                else:
                    op, ot, ol, oh = 'NA', 'NA', 'NA', 'NA'
                    lv, pt, po = 'NA', 'NA', 'NA'
                    st, sh, sa = 'NA', 'NA', 'NA'
                    iw, cl = 'NA', 'NA'

            else:
                # Non-verbal: all example-level variables → NA
                lv, pt, po = 'NA', 'NA', 'NA'
                st, sh, sa = 'NA', 'NA', 'NA'
                dv, cl, iw = 'NA', 'NA', 'NA'
                op, ot, ol, oh = 'NA', 'NA', 'NA', 'NA'

            # Append all values in column order
            hits.append(str(hit))
            help_form.append(help_word)
            help_pos.append(help_c5)
            voice_list.append(lv)
            prec_to_list.append(pt)
            polarity_list.append(po)
            subj_type_list.append(st)
            subj_head_list.append(sh)
            animacy_list.append(sa)
            dep_var.append(dv)
            comp_lemma.append(cl)
            interv_words.append(iw)
            obj_presence.append(op)
            obj_type_list.append(ot)
            obj_length_list.append(ol)
            obj_head_list.append(oh)
            file_id.append(id)
            year.append(yearvalue)
            genre.append(genrevalue)
            subgenre.append(subgenrevalue)
            mode.append(modevalue)
            variety.append("BrE")
            corpus.append("BNC")
            speaker_id.append(spk_id)
            speaker_gender.append(spk_gender)
            speaker_age.append(spk_age)
            soc.append(spk_soc)

            hit += 1
            collecting_right = True
            right_context = []
            current_right_context_words = 0

    # Store right context of the last hit in this file
    if collecting_right:
        right_KWIC = ' '.join(filter(None, right_context))
        help_KWIC.append(f"{left_KWIC} @{help_word}@ {right_KWIC}")

# --- (3) Build DataFrame and export ---

import pandas as pd

df_all = pd.DataFrame({
    'Hit':           hits,
    'KWIC':          help_KWIC,
    'Form':          help_form,
    'POS':           help_pos,
    'DepVar':        dep_var,
    'CompLemma':     comp_lemma,
    'IntervWords':   interv_words,
    'ObjPresence':   obj_presence,
    'ObjType':       obj_type_list,
    'ObjLength':     obj_length_list,
    'ObjHead':       obj_head_list,
    'Voice':         voice_list,
    'PrecedingTo':   prec_to_list,
    'Polarity':      polarity_list,
    'SubjType':      subj_type_list,
    'SubjHead':      subj_head_list,
    'Animacy':       animacy_list,
    'TextID':        file_id,
    'Year':          year,
    'Genre':         genre,
    'Subgenre':      subgenre,
    'Mode':          mode,
    'Variety':       variety,
    'Corpus':        corpus,
    'SpeakerID':     speaker_id,
    'SpeakerGender': speaker_gender,
    'SpeakerAge':    speaker_age,
    'SpeakerSoc':    soc,
})

df_written = df_all[df_all['Mode'] == 'Written'][[
    'Hit', 'KWIC', 'Form', 'POS', 'DepVar', 'CompLemma', 'IntervWords',
    'ObjPresence', 'ObjType', 'ObjLength', 'ObjHead',
    'Voice', 'PrecedingTo', 'Polarity',
    'SubjType', 'SubjHead', 'Animacy',
    'TextID', 'Year', 'Genre', 'Subgenre', 'Mode', 'Variety', 'Corpus'
]].reset_index(drop=True)

df_spoken = df_all[df_all['Mode'] == 'Spoken'][[
    'Hit', 'KWIC', 'Form', 'POS', 'DepVar', 'CompLemma', 'IntervWords',
    'ObjPresence', 'ObjType', 'ObjLength', 'ObjHead',
    'Voice', 'PrecedingTo', 'Polarity',
    'SubjType', 'SubjHead', 'Animacy',
    'TextID', 'SpeakerID', 'Year', 'SpeakerGender', 'SpeakerAge', 'SpeakerSoc',
    'Mode', 'Variety', 'Corpus'
]].reset_index(drop=True)

df_written.to_excel('BNC_help_written.xlsx', index=False)
df_spoken.to_excel('BNC_help_spoken.xlsx', index=False)

print(f"Written: {len(df_written)} results → BNC_help_written.xlsx")
print(f"Spoken:  {len(df_spoken)} results → BNC_help_spoken.xlsx")


/var/folders/yt/4l1xrk6s3bz2t7j232zb03fw0000gn/T/ipykernel_79291/2173684554.py:73: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  wstext_element = tree.find('.//wtext') or tree.find('.//stext')


Written: 52761 results → BNC_help_written.xlsx
Spoken:  4723 results → BNC_help_spoken.xlsx
